# 📓 Notebook 13 — Statistics That Pay for Themselves

> **Module:** Data Science · **Estimated time:** 55–75 min · **Difficulty:** Beginner / Intermediate

Most "data work" requires very little formal statistics — but the bit you *do* need pays for itself within a week. This notebook teaches the smallest set of statistical concepts that lets you:

- Describe a dataset with the *right* summary numbers.
- Tell whether a result is meaningful — not just whether it's *different*.
- Run an honest A/B test of two LLM providers.
- Express uncertainty when you report a number.

We will *not* prove theorems. We will use them, carefully, on the kind of data this course generates.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Pick the right summary statistic (mean / median / robust stats) for a distribution.
2. Read a **histogram + box plot** and explain what they say about a distribution's shape and tails.
3. Compute a **confidence interval** for a mean by hand and with a library.
4. Run a **t-test** (two-sample, two-sided) and *correctly* interpret its p-value.
5. Compute and report an **effect size** (Cohen's *d*) — the thing managers actually care about.
6. Calculate the **sample size** needed to detect a chosen effect.
7. Recognise the three classical statistical mistakes that ruin AI A/B tests in production.

## ✅ Prerequisites

NumPy (NB 11), matplotlib (NB 12). Comfort with means and standard deviations.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

RNG = np.random.default_rng(seed=42)


## 2. Two latency datasets — see them before you summarise them

We will use two simulated batches of LLM call latencies (in ms) throughout the notebook:

- **Model A** — a *fast* model. Mean ~1800 ms.
- **Model B** — a *slow* model with occasional spikes. Mean ~2400 ms.

In [ ]:
# Two latency distributions (simulated). Model B has occasional spikes.
n = 200
latency_a = RNG.normal(loc=1800, scale=300, size=n).clip(min=200)

# Mixture: 90% normal, 10% slow outliers
mask = RNG.random(n) < 0.9
latency_b = np.where(mask,
                     RNG.normal(2300, 350, n),
                     RNG.normal(4500, 600, n)).clip(min=200)

print(f"Model A:  n={len(latency_a)}, mean={latency_a.mean():.0f}, std={latency_a.std(ddof=1):.0f}")
print(f"Model B:  n={len(latency_b)}, mean={latency_b.mean():.0f}, std={latency_b.std(ddof=1):.0f}")


### Visualise — *always* before you summarise

A single chart often tells you more than a table of summary statistics. Histograms show the shape; box plots show the centre and tails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (1) Histograms
ax = axes[0]
ax.hist(latency_a, bins=30, alpha=0.6, label="Model A", color="#4C72B0", edgecolor="black")
ax.hist(latency_b, bins=30, alpha=0.6, label="Model B", color="#DD8452", edgecolor="black")
ax.axvline(latency_a.mean(), color="#4C72B0", lw=2, ls="--")
ax.axvline(latency_b.mean(), color="#DD8452", lw=2, ls="--")
ax.set_title("Histograms with means")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.legend()

# (2) Box plots
ax = axes[1]
bp = ax.boxplot([latency_a, latency_b], tick_labels=["Model A", "Model B"],
                patch_artist=True)
for patch, c in zip(bp["boxes"], ["#4C72B0", "#DD8452"]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_title("Box plots — centre & tails at a glance")
ax.set_ylabel("Latency (ms)")
plt.tight_layout(); plt.show()


**Reading what you see:**

- Model A is a clean bell-curve. Mean ≈ median, light tails, no outliers.
- Model B has a **right-skewed** distribution. The mean is dragged up by the spikes. The box plot shows those spikes as outlier dots above the upper whisker.

> 🎯 **The first statistical decision of any analysis** is whether the mean is a fair summary. When the distribution is skewed or has fat tails, the median is often more honest.

## 3. Mean vs median — when each one lies

In [ ]:
for name, data in [("Model A", latency_a), ("Model B", latency_b)]:
    print(f"{name}:")
    print(f"  mean   = {np.mean(data):>7.1f} ms")
    print(f"  median = {np.median(data):>7.1f} ms")
    print(f"  std    = {np.std(data, ddof=1):>7.1f} ms")
    p99 = np.percentile(data, 99)
    print(f"  p99    = {p99:>7.1f} ms     (top 1% of users wait this long or longer)")
    print()


**Why this matters operationally.** If you tell your team "Model B's average latency is 2.5 seconds" but its **p99 latency is 5 seconds**, you're under-reporting the real user experience for 1% of your traffic.

> 💡 **Always report p50, p95, p99 alongside the mean** for any latency / cost / waiting-time metric. The "tail" is where customers leave.

## 4. Confidence intervals — how sure are you of that mean?

The mean of a sample is a *point estimate*. A different sample would have given a slightly different mean. A **confidence interval** quantifies that wiggle room.

```
   95% CI for the mean of Model A:  [1761 ms, 1845 ms]
                                     │              │
                                     │              └─ upper bound
                                     └────────────── lower bound
```

The 95% CI says: *"if I repeated this experiment many times, 95% of the constructed intervals would contain the true mean."*

(That's the technically correct interpretation. Casually, "we're 95% sure the true mean is in here" is close enough for most business conversations.)

In [ ]:
def ci_of_mean(data, confidence=0.95):
    """Return (lower, upper) of the confidence interval for the mean."""
    n = len(data)
    mean = np.mean(data)
    sem  = np.std(data, ddof=1) / np.sqrt(n)        # standard error of the mean
    # Use the t-distribution — slightly wider than normal for small n
    crit = stats.t.ppf(0.5 + confidence/2, df=n-1)
    return mean - crit*sem, mean + crit*sem


for name, data in [("Model A", latency_a), ("Model B", latency_b)]:
    lo, hi = ci_of_mean(data, confidence=0.95)
    print(f"{name}:  mean = {data.mean():.1f} ms,  95% CI = [{lo:.1f}, {hi:.1f}]  "
          f"(±{(hi-lo)/2:.1f})")


**Three things to notice:**

1. Both CIs have a "half-width" of around 40–70 ms — the *uncertainty* on a 200-point mean.
2. The two CIs don't overlap → very strong evidence the means are *actually* different.
3. **Larger samples → narrower CIs.** Halving the half-width takes 4× the data.

> 🎯 **Rule of thumb for fast back-of-envelope CIs:** for n=200 normal-ish data, the half-width is roughly the standard deviation ÷ 7. (Comes from `1.96/sqrt(200) ≈ 0.14`.)

## 5. The t-test — is the difference *real*?

When you measure two things (Model A, Model B) and see different means, you need to know whether the difference is **bigger than the noise**. The classic tool is **Welch's t-test** (the safer of the two t-test flavours — doesn't assume equal variances).

In [ ]:
# Two-sample, two-sided Welch's t-test
result = stats.ttest_ind(latency_a, latency_b, equal_var=False)
print(f"t-statistic = {result.statistic:.2f}")
print(f"p-value     = {result.pvalue:.3e}")

# Mean difference and its 95% CI
diff = latency_b.mean() - latency_a.mean()
se = np.sqrt(np.var(latency_a, ddof=1) / len(latency_a) +
             np.var(latency_b, ddof=1) / len(latency_b))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"\nMean difference (B − A) = {diff:.1f} ms")
print(f"95% CI for the difference: [{ci_lo:.1f}, {ci_hi:.1f}]")


### How to read the output

- **t-statistic ≈ 24** means the difference of means is 24 standard errors away from zero — *enormous*.
- **p-value ≈ 10⁻⁶⁰** means: *"if the two models actually had the same mean latency, we'd see this big a difference by chance in fewer than 1 in 10⁶⁰ experiments."*
- **The CI for the difference excludes 0** → the difference is real.

> ⚠️ **Common p-value misunderstandings:**
>
> - p < 0.05 does **not** mean "95% chance the result is real" (a Bayesian probability).
> - p < 0.05 does **not** mean "the difference is important" — it might be 2 ms, statistically significant, and operationally meaningless.
> - p < 0.05 is **not** a free pass: with enough data, *every* tiny difference becomes "significant".
>
> The cure for all three: **always report effect size alongside p-value**.

## 6. Effect size — the number managers care about

The p-value tells you whether a difference is real. **Cohen's *d*** tells you how *big* the difference is, in units of the natural spread of the data.

$$d = \frac{\bar{x}_B - \bar{x}_A}{s_\text{pooled}}$$

| Cohen's *d* | Rough label |
|---|---|
| 0.2 | small |
| 0.5 | medium |
| 0.8 | large |
| > 1.0 | very large |

In [ ]:
def cohens_d(x, y):
    """Cohen's d for two independent samples."""
    nx, ny = len(x), len(y)
    pooled = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(y) - np.mean(x)) / pooled


d = cohens_d(latency_a, latency_b)
print(f"Cohen's d = {d:+.2f}")
print(f"Mean diff = {latency_b.mean() - latency_a.mean():.0f} ms")


**How to report this to a manager:**

> *"Model B is roughly 500 ms slower than Model A on average (Cohen's d ≈ 1.4, very large effect). 95% CI for the difference: [460, 560] ms. p < 10⁻⁵⁹."*

The numbers in **bold below** are the ones the manager will remember:

- **how big** is the difference (Cohen's d, mean diff)
- **how sure** are we (CI, p-value)
- **what does it mean in their units** (500 ms — would users notice?)

## 7. Sample size — how much data do I need?

Before you run a 10,000-user A/B test, ask: *"what's the smallest sample that would let me detect the effect I care about, with 80% power, at significance 0.05?"*

The formula (for two equal groups, two-sided t-test, normal data) is roughly:

$$n \approx \frac{16}{d^2}$$

per group, where `d` is the *minimum* effect size you'd care about.

In [ ]:
def sample_size_for_d(d, alpha=0.05, power=0.80):
    """Approximate per-group sample size to detect Cohen's d with given power."""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power)
    return int(np.ceil(2 * ((z_alpha + z_beta) / d) ** 2))


print(f"{'effect':>8}  {'per-group n':>12}")
for d in [0.1, 0.2, 0.5, 0.8, 1.0]:
    n = sample_size_for_d(d)
    print(f"  d = {d:.1f}    {n:>10,}")


**The big takeaway.** Tiny effects need *enormous* samples. If you can only run 200 users per arm, you can reliably detect a Cohen's d of about 0.3 — but anything smaller will hide in the noise.

> 💡 **Sample-size planning prevents the silent failure** where you ran a test, the p-value was 0.12, and you concluded "no effect" — when you actually just didn't have enough data.

## 8. A complete A/B-test report

The "good" version of A/B reporting puts all five numbers a manager wants on one screen:

In [ ]:
def ab_report(name_a, x, name_b, y, alpha=0.05):
    """Print a five-number A/B summary that's safe to send to a manager."""
    n_a, n_b   = len(x), len(y)
    mean_a, mean_b = np.mean(x), np.mean(y)
    diff       = mean_b - mean_a
    se_diff    = np.sqrt(np.var(x, ddof=1)/n_a + np.var(y, ddof=1)/n_b)
    z_crit     = stats.norm.ppf(1 - alpha/2)
    ci_lo, ci_hi = diff - z_crit*se_diff, diff + z_crit*se_diff
    d          = cohens_d(x, y)
    p_val      = stats.ttest_ind(x, y, equal_var=False).pvalue

    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" A/B test: {name_a}  vs  {name_b}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f" Sample sizes      : {n_a} vs {n_b}")
    print(f" Mean              : {mean_a:.1f}  vs  {mean_b:.1f}")
    print(f" Difference (B−A)  : {diff:+.1f}")
    print(f" 95% CI for diff   : [{ci_lo:+.1f}, {ci_hi:+.1f}]")
    print(f" Cohen's d         : {d:+.2f}     (effect size)")
    print(f" p-value           : {p_val:.2e}")
    verdict = "REAL difference" if p_val < alpha and abs(d) > 0.2 \
              else "NOT meaningfully different"
    print(f" Verdict           : {verdict}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


ab_report("Model A", latency_a, "Model B", latency_b)


**Why this template works.**

- **Sample sizes** show whether the test was powered enough to trust the verdict.
- **Means + difference + CI** answer "by how much?" — the manager's first question.
- **Cohen's d** answers "is the difference *big* in real-world terms?" — the second question.
- **p-value** answers "is it real?" — the third question.
- The **verdict** combines the two ("real *and* meaningful"), preventing the common mistake of acting on a statistically significant but operationally tiny effect.

## 9. The three classical mistakes (and the fixes)

| Mistake | What happens | Fix |
|---|---|---|
| **Peeking** — running a test, looking at the p-value daily, stopping the first time it dips below 0.05 | False positives explode. With 5 peeks the *actual* false-positive rate is around 20%, not 5%. | Plan the sample size in advance. Stop *only* at that pre-planned moment. |
| **Multiple comparisons** — running 20 t-tests on 20 metrics, declaring the one with p<0.05 as a "win" | At alpha=0.05 you expect 1 false positive per 20 tests by definition. | Use Bonferroni (alpha/k) or false-discovery-rate correction. Or: pre-register your hypothesis. |
| **Reporting only p, not d** | Statistically significant but operationally meaningless effects get shipped. | Always report effect size alongside p-value. |

## 🧪 Practice exercises

### Exercise 1 — Robust statistics on a heavy-tailed metric

Compute the **mean**, **median**, **5%-trimmed mean**, and **standard deviation** of `latency_b`. Which one would you report to a manager who asks "what's a typical latency?".

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from scipy.stats import trim_mean

print(f"mean              = {np.mean(latency_b):.1f}")
print(f"median            = {np.median(latency_b):.1f}")
print(f"5%-trimmed mean   = {trim_mean(latency_b, 0.05):.1f}")
print(f"std               = {np.std(latency_b, ddof=1):.1f}")
```

The **median** is usually the most honest "typical user" number on a right-skewed
metric like latency. The mean and std are dragged up by the slow tail. The
trimmed mean (drop the top and bottom 5% before averaging) is a robust compromise.
</details>

### Exercise 2 — Confidence interval, by hand

Without using `stats.t.ppf`, compute an *approximate* 95% CI for the mean of `latency_a` using the rule of thumb `mean ± 1.96 × std/√n`. Compare with the t-distribution-based CI from §4.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
n = len(latency_a)
mean = latency_a.mean()
sem  = latency_a.std(ddof=1) / np.sqrt(n)

print(f"Normal-approx CI :  [{mean - 1.96*sem:.1f},  {mean + 1.96*sem:.1f}]")
print(f"t-distribution CI:  [{ci_of_mean(latency_a)[0]:.1f},  {ci_of_mean(latency_a)[1]:.1f}]")
```

For n=200 the two are almost identical (the t-distribution converges to normal as n grows). For n<30 the t-distribution CI is noticeably wider — which is exactly why people invented it.
</details>

### Exercise 3 — Sample size planning

You want to detect a **0.3 effect size** with 90% power and significance 0.05. How many users per arm do you need? What if you can tolerate 80% power instead?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
print(f"d = 0.3, power = 0.80 → n = {sample_size_for_d(0.3, power=0.80):,} per arm")
print(f"d = 0.3, power = 0.90 → n = {sample_size_for_d(0.3, power=0.90):,} per arm")
print(f"d = 0.5, power = 0.80 → n = {sample_size_for_d(0.5, power=0.80):,} per arm")
```

Doubling the power from 80% to 90% requires ~30% more data — a real cost. The
practical takeaway: 80% power is the standard convention for a reason; pushing
to 90% costs a lot of users.
</details>

### Exercise 4 — Debug me 🐞

The snippet below is supposed to compute the standard error of the mean, but the answer is consistently a factor of √n too small. Find the bug.

```python
def sem_buggy(data):
    return np.std(data, ddof=1) * np.sqrt(len(data))
```

In [ ]:
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

The formula for the *standard error of the mean* is `σ / √n`, not `σ × √n`.

```python
def sem(data):
    return np.std(data, ddof=1) / np.sqrt(len(data))
```

Sanity check: as `n` grows, the standard error should *shrink* (we're more sure of the mean). The buggy version grows instead — the immediate signal that the formula is upside down.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Paired-samples t-test

Simulate 30 customers measured *before* and *after* a UX change. The change made each customer ~5% more satisfied. Run a **paired-samples t-test** (`stats.ttest_rel`) and compare with the unpaired version. Which is more powerful, and why?


In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(0)
before = rng.normal(3.8, 0.5, 30)
after  = before * 1.05 + rng.normal(0, 0.05, 30)

paired   = stats.ttest_rel(before,  after)
unpaired = stats.ttest_ind(before,  after, equal_var=False)

print(f"Paired   t = {paired.statistic:.2f}, p = {paired.pvalue:.4f}")
print(f"Unpaired t = {unpaired.statistic:.2f}, p = {unpaired.pvalue:.4f}")
```

**Why paired is more powerful.** When the same *customer* is
measured twice, customer-level baseline noise cancels out. The
paired test removes that noise from the denominator → larger
t-statistic → smaller p-value for the same true effect. Use the
paired test whenever you have natural pairs (before/after,
A vs B on the same query, etc.).

</details>

### Stretch exercise B — Bootstrap confidence interval for the median

The formulas in this notebook give CIs for the *mean*. Implement a **bootstrap CI for the median** (resample with replacement, take the 2.5th / 97.5th percentile of the bootstrap medians). Test on a heavy-tailed sample.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def boot_ci_median(data, n_boot=2000, ci=0.95, rng=None):
    rng = rng or np.random.default_rng(0)
    n   = len(data)
    meds = np.array([np.median(rng.choice(data, n, replace=True))
                      for _ in range(n_boot)])
    alpha = (1 - ci) / 2
    return float(np.median(data)), float(np.quantile(meds, alpha)), float(np.quantile(meds, 1 - alpha))


# Heavy-tailed sample
sample = np.concatenate([np.random.default_rng(0).normal(2, 0.5, 95),
                          np.random.default_rng(1).normal(8, 1.0, 5)])
point, lo, hi = boot_ci_median(sample)
print(f"Median = {point:.2f}   95% CI = [{lo:.2f}, {hi:.2f}]")
```

**Bootstrap is the Swiss-army knife of statistics.** No formula
needed — just resample and measure. Same approach works for any
statistic: trimmed mean, 95th percentile, Gini coefficient, etc.

</details>

## 🎁 Bonus mini-project — Power analysis dashboard

Write a function `power_table(d_values, n_values)` that returns a DataFrame showing the statistical power (probability of detecting an effect of size `d` at significance 0.05) for every combination of effect size and sample size.

Use it to answer: "what's the smallest sample I need to have 80% power for a 0.5 effect size?".

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def power_for(d, n, alpha=0.05):
    """Approximate power of a two-sample, two-sided t-test."""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    # Non-centrality parameter
    ncp = d * np.sqrt(n / 2)
    return 1 - stats.norm.cdf(z_alpha - ncp) + stats.norm.cdf(-z_alpha - ncp)


def power_table(d_values, n_values):
    rows = []
    for d in d_values:
        rows.append({"d": d,
                     **{f"n={n}": round(power_for(d, n), 2) for n in n_values}})
    return pd.DataFrame(rows).set_index("d")


tbl = power_table([0.1, 0.2, 0.3, 0.5, 0.8],
                  [50, 100, 200, 500, 1000])
print(tbl)
```

You can read the table directly: each cell is the probability that you'd
detect the effect *if it's really there*. Use it to plan experiments without
running 100 simulations.
</details>

## 🧠 Key takeaways

1. **Plot before you summarise.** Histograms and box plots tell you the shape; means and standard deviations don't.
2. For **skewed or heavy-tailed** data, report median and percentiles (p50, p95, p99) *alongside* the mean.
3. A **confidence interval** turns a point estimate into honest uncertainty. The half-width shrinks with √n, so doubling precision needs 4× the data.
4. A **t-test** answers "is the difference real?". An **effect size** (Cohen's *d*) answers "is it big?". You need both.
5. **Sample-size planning** prevents the silent failure of an underpowered test.
6. The three classical mistakes are **peeking, multiple comparisons, and reporting only p**. The fixes are pre-registration, correction, and effect size.
7. Every A/B report you ship should have: sample sizes, means, difference + CI, Cohen's *d*, p-value, *and* a verdict.

## ✅ Self-assessment

- [ ] Plot a histogram + box plot and explain skewness, tails, outliers
- [ ] Compute mean, median, p95, p99 for a sample
- [ ] Compute a 95% CI for a mean
- [ ] Run a Welch's t-test and correctly interpret the p-value
- [ ] Compute and interpret Cohen's d
- [ ] Plan a sample size for a target effect size and power
- [ ] Spot the three classical A/B mistakes

## 🚀 Next step

Continue with **Notebook 14 — Time Series and Forecasting**, where the statistics you just learned become the backbone of evaluating forecasts honestly.